# 03a — Preparación no supervisada (conjuntos A/B/C)

Ejecuta el §5 + primera comprobación §7A de planning/plan_unsupervised_implementation.md.
Nota de convención: las transformaciones de ≤5 líneas viven aquí por legibilidad; todo lo que puede romperse en silencio (listas, matemática CLC, guardianes, config) vive en src/wildfire/processing/unsupervised.py con tests en 	ests/test_unsupervised.py.
Entrada: data/processed/expanded/firms_spain_final_lags_index.csv (la forma exacta la imprime la primera celda de código). Nunca irms_spain_final_normalized.csv (scaler descriptivo, plan §4 regla 3).
Salida: data/processed/unsupervised/features_{a,b,c}.csv + perfilado.csv (índice común).

## 03a.0 — Imports (celda única del cuaderno)

Todos los imports viven aquí y solo aquí: el resto de celdas no importa nada (convención del repo). Están la ciencia de datos habitual (`numpy`, `pandas`, `sklearn`), la E/S anclada a la raíz (`load_csv`/`save_csv`) y el módulo del bloque (`processing/unsupervised.py`), de donde salen las listas, las transformaciones y el guardián.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import RobustScaler, StandardScaler
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.pca_weather import WEATHER_PCA_BASE, fit_weather_pca
from wildfire.processing.unsupervised import (
    ENTRADA_A, ENTRADA_B, ENTRADA_C_FIXED, PERFILADO, SEASON_SINCOS, TIME_COL,
    add_wind_circular, assert_feature_sets_clean,
    build_clc_environment, clc_environment_columns, resolve_set,
)


## 03a.1 — Cargar base, derivar viento circular y ajustar el PCA meteorológico

Primero se carga `firms_spain_final_lags_index.csv`, la base de todo el bloque 03 (forma impresa en la salida). Después se deriva el viento circular (`wd_sin`/`wd_cos` con `AddWindCircular`, igual que en el PCA) y se comprueba que el par existe. La dirección cruda no entra en ningún conjunto: 359° y 1° son casi lo mismo y un escalado ingenuo rompería esa cercanía.

Aquí mismo se ajusta el PCA meteorológico (`fit_weather_pca`, la misma solución de 7 PCs y ~90,9 % de varianza de 02b): el conjunto A pasa a ser las puntuaciones PC1–PC7 en vez de las 14 variables crudas. Motivo (plan §5, variante recomendada): las crudas traen pares muy correlacionados (temperatura↔VPD, capas de humedad de suelo, ET0↔radiación) y las distancias euclídeas de KMeans/GMM las ponderarían dos veces; los PCs están decorrelacionados por construcción. El pipeline lleva dentro su propio preprocesado (viento circular, log1p de precipitación, imputación por mediana, z-score), así que se ajusta sobre la base cruda.


In [2]:
df = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
print(f"{len(df):,} rows × {df.shape[1]} columns")
df = add_wind_circular(df)
assert {"wd_sin", "wd_cos"} <= set(df.columns)
pca_result = fit_weather_pca(df)
pc_scores = pca_result.scores
print(f"PCA meteorológico: {pca_result.n_components} PCs "
      f"({100.0 * pca_result.variance_ratio[:pca_result.n_components].sum():.1f} % de varianza, solución de 02b)")


23,142 rows × 60 columns
PCA meteorológico: 7 PCs (91.5 % de varianza, solución de 02b)


## 03a.2 — Mostrar los conjuntos cerrados y vigilarlos

Las listas A/B/C y PERFILADO se imprimen aquí para leer el pipeline entero sin abrir `src/`. A ya no es una lista de columnas crudas: son las puntuaciones PC1–PCk del PCA ajustado en 03a.1 (las listas `ENTRADA_*` del módulo describen el contenido conceptual y alimentan el guardián; el conjunto D, espacial puro, se difiere a un futuro 03d y no se construye aquí). Después pasa el guardián: si alguna columna de satélite (`frp`, `brightness`…) se hubiera colado en las entradas, la celda falla en cerrado en vez de contaminar el clustering. El contrato queda así: las entradas son causas cotidianas; el satélite y los presupuestos solo sirven para perfilar.


In [3]:
_b_extras = [*SEASON_SINCOS, TIME_COL]
print(f"A ({len(pc_scores.columns)}): {list(pc_scores.columns)}")
print(f"B ({len(pc_scores.columns) + len(_b_extras)}): A + {_b_extras}")
print("C: B + ['clc_class', 'pct_vecinos_igual_centro', 'n_clases_distintas_vecinos'] + freq_vecina_* (entorno construido en 03a.3)")
print(f"PERFILADO ({len(PERFILADO)})")
assert_feature_sets_clean(ENTRADA_A, ENTRADA_B, ENTRADA_C_FIXED)
print("Guardián OK: sin columnas de satélite en las entradas.")


A (7): ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7']
B (12): A + ['month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'franja_horaria']
C: B + ['clc_class', 'pct_vecinos_igual_centro', 'n_clases_distintas_vecinos'] + freq_vecina_* (entorno construido en 03a.3)
PERFILADO (31)
Guardián OK: sin columnas de satélite en las entradas.


## 03a.3 — Entorno CLC (transformación de ≤5 líneas, aquí por la nota de convención)

Las ocho direcciones CLC se colapsan en un entorno interpretable (% de vecinos iguales, nº de clases distintas, frecuencias) con el constructor que cubren los tests. Al final se enseña un resumen del entorno para inspeccionarlo a ojo.

(El `log1p` en precipitación, ráfagas y VPD que el plan §5 pedía para las 14 crudas ya no se aplica aquí: con el conjunto A como PCs, el de precipitación vive dentro del pipeline del PCA —`LogPrecipitation`— y ninguna salida usa ya esas columnas transformadas; ráfagas y VPD entran al PCA en bruto, exactamente igual que en la solución verificada de 02b.)


In [4]:
env = build_clc_environment(df)
df = pd.concat([df, env], axis=1)
env_cols = clc_environment_columns(env)
print(f"CLC environment: {len(env_cols)} columns ({len([c for c in env_cols if c.startswith('freq_vecina_')])} frequencies)")
display(env.describe().T[["mean", "std", "min", "max"]])


CLC environment: 12 columns (10 frequencies)

,mean,std,min,max
pct_vecinos_igual_centro,0.861037,0.221914,0.0,1.0
n_clases_distintas_vecinos,1.462925,0.668764,1.0,5.0
freq_vecina_1,0.008864,0.061032,0.0,1.0
freq_vecina_2,0.155615,0.338702,0.0,1.0
freq_vecina_3,0.110702,0.287268,0.0,1.0
freq_vecina_4,0.095146,0.262270,0.0,1.0
freq_vecina_5,0.289749,0.412100,0.0,1.0
freq_vecina_6,0.153795,0.305779,0.0,1.0
freq_vecina_7,0.117195,0.303282,0.0,1.0
freq_vecina_9,0.063786,0.214334,0.0,1.0


## 03a.4 — Resolver conjuntos y guardar (índice común)

Aquí se recorta el dataframe a cada conjunto (`resolve_set`: lo que falte avisa y se descarta, sin romper nada) y se guardan los cuatro ficheros en `data/processed/unsupervised/`: `features_a` (7, los PCs del clima), `features_b` (12, + estacionalidad y hora), `features_c` (25, + suelo) y `perfilado` (31: satélite, presupuestos e índices). Los cuatro comparten índice para poder re-unirlos en 03b–03e sin ambigüedad.


In [5]:
sets = {
    "features_a": pc_scores,
    "features_b": pd.concat(
        [pc_scores, resolve_set(df, [*SEASON_SINCOS, TIME_COL], "B-extra")], axis=1
    ),
    "features_c": pd.concat(
        [pc_scores,
         resolve_set(df, [*SEASON_SINCOS, TIME_COL, "clc_class", *env_cols], "C-extra")],
        axis=1,
    ),
    "perfilado": resolve_set(df, PERFILADO, "PERFILADO"),
}
for _name, _frame in sets.items():
    save_csv(_frame, f"data/processed/unsupervised/{_name}.csv")
    print(f"{_name}: {_frame.shape}")


features_a: (23142, 7)


features_b: (23142, 12)


features_c: (23142, 25)


perfilado: (23142, 31)


## 03a.5 — Primer número de robustez: ARI Standard vs Robust (plan §7A)

El primer número de robustez del bloque: agrupar lo mismo dos veces con KMeans (k=5, semilla 42), una con `StandardScaler` y otra con `RobustScaler`, sobre las 12 continuas meteorológicas, y medir el acuerdo con ARI (1 = idénticas, 0 = nada que ver). El plan exige más de 0,6 entre variantes. No es una elección de k —eso llega en 03b/03e—, sino una sonda: si saliera bajo, los grupos dependerían del scaler y habría que declararlo.

In [6]:
base12 = resolve_set(df, WEATHER_PCA_BASE, "BASE12")
std_labels = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(StandardScaler().fit_transform(base12))
rob_labels = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(RobustScaler().fit_transform(base12))
ari = adjusted_rand_score(std_labels, rob_labels)
print(f"ARI(Standard, Robust) @ k=5: {ari:.3f} (listón plan §7A: > 0,6 entre variantes)")

ARI(Standard, Robust) @ k=5: 0.698 (listón plan §7A: > 0,6 entre variantes)
